## set Price regime

In [3]:
from datasets import load_dataset
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

c:\Users\lenovo\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
df = pd.read_feather("../Outputs/01_df.feather")

In [ ]:

cat_conditions = [
    df["cat2_slug"].isin(["residential-sell", "commercial-sell"]),
    df["cat2_slug"].isin(["residential-rent", "commercial-rent"]),
    df["cat2_slug"] == "temporary-rent",
    df["cat2_slug"] == "real-estate-services"
]

cat_choices = [
    "sell",
    "credit&rent",
    "rent",
    "services"]

df["ad_type_by_cat"] = np.select(cat_conditions, cat_choices, default="نامشخص")

# -------------------------------------------------------------
# ۲. ساخت ستون بر اساس ستون‌های مالی و قیمت‌ها
# -------------------------------------------------------------
# تعریف شرایط وجود مقادیر عددی بزرگتر از صفر
has_price = df["price_value"].notna() & (df["price_value"] > 0)
has_rent = df["rent_value"].notna() & (df["rent_value"] > 0)
has_credit = df["credit_value"].notna() & (df["credit_value"] > 0)

# تعریف شرایط بر اساس حالت‌های قیمت (mode)
# (گاهی قیمت عددی صفر است ولی حالت قیمت مثلاً توافقی یا معاوضه است)
mode_sell = df["price_mode"].notna() & (df["price_mode"] != "none")
mode_rent = df["rent_mode"].notna() & (df["rent_mode"] != "مجانی")
mode_credit = df["credit_mode"].notna() & (df["credit_mode"] != "مجانی")

price_conditions = [
    # الف) شرایط فروش: قیمت دارد ولی اجاره و ودیعه ندارد
    (has_price | mode_sell) & ~(has_rent | mode_rent) & ~(has_credit | mode_credit),
    
    # ب) رهن کامل: فقط ودیعه/رهن دارد
    (has_credit | mode_credit) & ~(has_rent | mode_rent) & ~(has_price | mode_sell),
    
    # ج) رهن و اجاره: هم ودیعه و هم اجاره دارد
    (has_credit | mode_credit) & (has_rent | mode_rent) & ~(has_price | mode_sell),
    
    # د) اجاره کامل: فقط اجاره دارد (بدون ودیعه)
    (has_rent | mode_rent) & ~(has_credit | mode_credit) & ~(has_price | mode_sell)
]

price_choices = [
    "sell",
    "credit",
    "credit and rent",
    "rent"
]

df["ad_type_by_price"] = np.select(price_conditions, price_choices, default="unknown")


print(df["ad_type_by_cat"].value_counts(dropna=False))
print(df["ad_type_by_price"].value_counts(dropna=False))

In [ ]:
# فیلتر کردن ردیف‌های متناقض
mismatched_sell = df[
    (df["ad_type_by_cat"] == "sell") & 
    (df["ad_type_by_price"] != "sell")&
    (df["ad_type_by_price"] != "نامشخص")
]
# Remove those category = service

# چاپ تعداد کل موارد متناقض
print(f"تعداد آگهی‌های با دسته‌بندی فروش اما فاقد قیمت فروش: {len(mismatched_sell)}")

# نمایش ستون‌های کلیدی برای بررسی علت تناقض (مثلاً ۵ ردیف اول)
cols_to_show = [
    "title", 
    "cat2_slug", 
    "price_mode", 
    "price_value", 
    "rent_value", 
    "credit_value", 
    "ad_type_by_price",
    "ad_type_by_cat"
]
mismatched_sell[cols_to_show].head(20)
# mismatched_sell["ad_type_by_price"].unique()


In [ ]:
df["rent_mode"].unique()

In [ ]:
df.loc[
    df["cat2_slug"].str.contains("rent", na=False),
    ["cat2_slug", "rent_mode", "credit_mode", "price_value", "rent_value", "credit_value","ad_type_by_cat","ad_type_by_price"]
]
